# CosQA E5 + HyDE + re-ranking

This notebook is the ordered entry point for the four-system comparison: `e5`, `e5_hyde`, `e5_rerank`, and `e5_hyde_rerank`. It uses the pinned CosQA dataset, text-only E5 inputs, candidate depth 1,000, the declared qrels split, and official COIR `nDCG@10`. HyDE retains the original question and appends one generated hypothesis. The HyDE and cross-encoder outputs are each fused with the original E5 ranking using weights selected on the complete `valid` qrels split. The combined row fuses those two selected component rankings. Raw HyDE and reranker scores remain diagnostics; all settings, candidate-ID digests, and run provenance are saved.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

workspace_root = Path.cwd()
while workspace_root != workspace_root.parent and not (workspace_root / 'code-retrieval' / 'src').exists():
    workspace_root = workspace_root.parent
project_dir = workspace_root / 'code-retrieval'
if not (project_dir / 'src').exists():
    project_dir = Path.cwd()
    workspace_root = project_dir.parent
os.chdir(project_dir)
sys.path.insert(0, str(project_dir / 'src'))
sys.path.insert(0, str(project_dir / 'scripts'))
from run_benchmark_notebook import require_pinned_runtime

from e5_baseline import environment_metadata, package_versions, set_seed
from e5_hyde_rerank import (
    ExperimentConfig,
    blocked_result,
    load_cosqa,
    run_experiment,
    select_run_data,
    write_comparison_artifacts,
)

print('Project directory:', project_dir.resolve())
print('Python:', sys.version.split()[0])

## 1. Environment and explicit controls

The generator, re-ranker, E5 settings, dataset revision, seed, candidate depth, and execution mode are configuration, not hidden notebook state.

In [ ]:
RUN_MODE = os.environ.get('E5_HYDE_RERANK_MODE', 'smoke').strip().lower()
if RUN_MODE not in {'smoke', 'benchmark'}:
    raise ValueError('E5_HYDE_RERANK_MODE must be smoke or benchmark')
if RUN_MODE == 'benchmark':
    require_pinned_runtime()

batch_size = int(os.environ.get('E5_HYDE_RERANK_BATCH_SIZE', '4' if RUN_MODE == 'smoke' else '128'))
candidate_depth = 1000
if candidate_depth != 1000:
    raise RuntimeError('The primary comparison requires candidate depth 1000')
torch_threads = int(os.environ.get('E5_HYDE_RERANK_TORCH_THREADS', '8'))
if torch_threads > 0:
    import torch
    torch.set_num_threads(torch_threads)

qrels_split = os.environ.get('E5_HYDE_RERANK_QRELS_SPLIT', 'test')
hyde_fusion_weights = (0.65, 0.35)
reranker_fusion_weights = (0.8, 0.2)
combined_fusion_weights = (0.75, 0.25)
if RUN_MODE == 'benchmark' and qrels_split == 'test':
    selection_path = Path('artifacts/e5_hyde_rerank/validation/weight_selection.json')
    if not selection_path.is_file():
        raise RuntimeError('Run valid-split fusion selection before evaluating test qrels')
    selection = json.loads(selection_path.read_text(encoding='utf-8'))
    if selection.get('selection_qrels_split') != 'valid' or selection.get('query_count') != 500 or selection.get('candidate_depth') != 1000:
        raise RuntimeError('Test fusion weights must come from the complete valid split')
    hyde_fusion_weights = tuple(selection['systems']['e5_hyde']['selected']['weights'])
    reranker_fusion_weights = tuple(selection['systems']['e5_rerank']['selected']['weights'])
    combined_fusion_weights = tuple(selection['systems']['e5_hyde_rerank']['selected']['weights'])

config = ExperimentConfig(
    run_mode=RUN_MODE,
    qrels_split=qrels_split,
    batch_size=batch_size,
    candidate_depth=candidate_depth,
    reranker_batch_size=batch_size,
    cache_dir='artifacts/e5_hyde_rerank/cache',
    artifact_dir='artifacts/e5_hyde_rerank',
    hyde_fusion_weights=hyde_fusion_weights,
    reranker_fusion_weights=reranker_fusion_weights,
    combined_fusion_weights=combined_fusion_weights,
)
set_seed(config.seed)
print(json.dumps(config.as_dict(), indent=2, sort_keys=True))
print('Installed packages:')
print(json.dumps(package_versions(['coir-eval', 'datasets', 'faiss-cpu', 'numpy', 'pytrec-eval-terrier', 'sentence-transformers', 'torch', 'transformers']), indent=2, sort_keys=True))
print('Hardware/runtime:')
environment = environment_metadata(config, repo_root=workspace_root)
print(json.dumps(environment, indent=2, sort_keys=True, default=str))

## 2. Load and inspect the real CosQA schema

The loader reads the separate corpus, query, and test-qrels configurations from one pinned dataset revision before any model call. Titles and unrelated fields are not passed into E5 or the cross-encoder.

In [ ]:
data = load_cosqa(config)
print(json.dumps(data.schema, indent=2, sort_keys=True, default=str))
print({
    'corpus_count': len(data.corpus),
    'query_count_with_test_qrels': len(data.queries),
    'qrels_query_count': len(data.qrels),
    'qrels_judgment_count': sum(len(rels) for rels in data.qrels.values()),
})
print('Corpus example:', next(iter(data.corpus.items())))
print('Query example:', next(iter(data.queries.items())))
print('Qrels example:', next(iter(data.qrels.items())))

## 3. Select the evidence level

Smoke mode uses a small labeled subset and records exclusions. Benchmark mode must use every declared test-qrels query and every corpus document.

In [ ]:
run_data = select_run_data(data, config)
if config.run_mode == 'benchmark' and (len(run_data.queries) != len(data.queries) or len(run_data.corpus) != len(data.corpus)):
    raise RuntimeError('benchmark mode must use the complete declared CosQA test queries and corpus')
print(json.dumps({
    'run_mode': config.run_mode,
    'query_count': len(run_data.queries),
    'corpus_count': len(run_data.corpus),
    'qrels_query_count': len(run_data.qrels),
    'qrels_judgment_count': sum(len(rels) for rels in run_data.qrels.values()),
    'exclusions': run_data.exclusions,
}, indent=2))

## 4. Run the controlled four-system experiment

The reusable runner loads real E5, HyDE, and cross-encoder models, resolves their revisions, validates each cache identity, and evaluates all systems against the same qrels. The original query remains the evaluation query; only the model input representation changes for the HyDE systems. Re-ranking is scored in one batched pass per candidate collection, and the final combined ranking is a deterministic weighted reciprocal-rank fusion that retains the configured 1,000-document depth.

In [ ]:
notebook_path = project_dir / 'notebooks' / 'e5_hyde_rerank_experiment.ipynb'
notebook_sha256 = hashlib.sha256(notebook_path.read_bytes()).hexdigest() if notebook_path.exists() else None
environment = environment_metadata(config, repo_root=workspace_root)

try:
    result, artifact_paths = run_experiment(
        config,
        repo_root=workspace_root,
        notebook_sha256=notebook_sha256,
    )
except Exception as error:
    blocker = f'{type(error).__name__}: {error}'
    result = blocked_result(
        config,
        blocker=blocker,
        environment=environment,
        repo_root=workspace_root,
        notebook_sha256=notebook_sha256,
    )
    artifact_paths = write_comparison_artifacts(config, result)
    print('Execution blocker recorded:', blocker)

print(json.dumps({
    'status': result['status'],
    'benchmark_evidence': result['benchmark_evidence'],
    'system_ids': result['system_ids'],
    'artifact_paths': artifact_paths,
}, indent=2))

## 5. Inspect the comparison and interpretation boundary

The table is produced from evaluator output, not entered by hand. Deltas are absolute changes versus the same-run E5 row. Smoke values prove wiring only, and blocked values are null rather than partial benchmark evidence.

In [ ]:
print(json.dumps(result['results'], indent=2, sort_keys=True))
print(json.dumps({
    'metric': result['metric'],
    'status': result['status'],
    'benchmark_evidence': result['benchmark_evidence'],
    'candidate_depth': result['candidate_depth'],
    'dataset_revision': result['dataset']['revision'],
    'hyde': result['hyde'],
    'reranker': result['reranker'],
    'fusion': result['fusion'],
    'component_diagnostics': result['component_diagnostics'],
    'candidate_contract': result.get('candidate_contract', {}),
    'exclusions': result['exclusions'],
    'limitations': result['limitations'],
    'provenance': result['artifact_provenance'],
}, indent=2, sort_keys=True, default=str))